Clonar el repo
git clone https://<username>:<PAT>@github.com/<owner>/<repo>.git

In [ ]:
User=""
PAT="" # Pat = personal access token se sugiere descarglo como .env o duplicar este archivo
url = f"https:/{User}:{PAT}@github.com/Fabricio552/Trabajo-de-sistemas-intro.-a-la-ciencia-de-datos"
!git clone url

In [ ]:
#import del sistema
import pandas as pd
import os
#import third party
from google.colab import files
import gdown

In [ ]:
!mkdir "/content/mi_proyecto"
!mkdir "/content/mi_proyecto/datos"

In [ ]:
#_____ Train dataframe
file_id = "1hFv2KUgOi64nQPaO5ybx-G8CZHHs7sqD"
output = "/content/mi_proyecto/datos/train.csv"
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

In [ ]:
#_____ Train series dataframe
file_id = "1QxxEDx-yiN02p_yOOJ4YK4rd2xdDGWp2"
output = "/content/mi_proyecto/datos/train_series.csv"
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

In [ ]:
train = pd.read_csv("/content/mi_proyecto/datos/train.csv")

In [ ]:
train_series_csv = pd.read_csv("/content/mi_proyecto/datos/train_series.csv")

In [ ]:
train.describe(include="all")

In [ ]:
train_series_csv.describe(include="all")

In [ ]:
train_series_csv.info()

In [ ]:
train.info()

In [ ]:
train["ACL"].unique()

In [ ]:
#Cambiamos los nulos por 2 ya que las variables son binarias 2 = es negativo
train = train.fillna(2)

In [ ]:
train.describe(include = "all")

In [ ]:
train.info()

In [ ]:
df_display = train.isna()== True

In [ ]:
df_display

In [ ]:
train_etiquetas = train[train["ACL"] != 2]
train_etiquetas.head(58)

TEXT EMBEDDING GRAPH FRACTURA

In [ ]:
!pip install openai tqdm langdetect numpy

In [ ]:
train_etiquetas_fractura = train_etiquetas[train_etiquetas["Fracture"] == 1]

In [ ]:
train_etiquetas_fractura.reset_index(drop=True, inplace=True)

In [ ]:
import json
import re
from langdetect import DetectorFactory, detect
import numpy as np
from openai import OpenAI
import pandas as pd
from tqdm import tqdm

DetectorFactory.seed = 0

# Recuerda usar tu NUEVA API key regenerada
client = OpenAI(api_key="smgA") # Truncada usen la suya
MODEL_NAME = "text-embedding-3-large"


def detect_language_safe(text):
    try:
        return detect(text[:500])
    except Exception:
        return "en"


def split_sentences(text):
    sentences = re.split(r"[\n\.;]+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 8]


# 1. Cargar datos (prueba con los primeros registros o todo el df)
df = train_etiquetas_fractura # Cambia a [:1] o todo df según necesites
col_reportes = df.columns[1]  # Toma la primera columna

print(f"Iniciando extracción de embeddings para {len(df)} reportes...")

records = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    report_text = str(row[col_reportes])
    lang = detect_language_safe(report_text)
    sentences = split_sentences(report_text)

    # Si no hay texto u oraciones válidas
    if not sentences:
        records.append({
            "idx_original": idx,
            "idioma": lang,
            "n_oraciones": 0,
            "oraciones_json": json.dumps([]),
            "embeddings_oraciones_json": json.dumps([]),
            "embedding_promedio_json": json.dumps([]),
        })
        continue

    try:
        # Llamada a la API para obtener los embeddings de todas las frases del informe
        resp = client.embeddings.create(input=sentences, model=MODEL_NAME)
        sentence_embeddings = [item.embedding for item in resp.data]

        # Vector centroide (promedio) de todo el informe
        avg_embedding = np.mean(sentence_embeddings, axis=0).tolist()

        records.append({
            "idx_original": idx,
            "idioma": lang,
            "n_oraciones": len(sentences),
            "oraciones_json": json.dumps(sentences, ensure_ascii=False),
            "embeddings_oraciones_json": json.dumps(sentence_embeddings),
            "embedding_promedio_json": json.dumps(avg_embedding),
        })

    except Exception as e:
        print(f"Error procesando el índice {idx}: {e}")
        records.append({
            "idx_original": idx,
            "idioma": lang,
            "n_oraciones": len(sentences),
            "oraciones_json": json.dumps([]),
            "embeddings_oraciones_json": json.dumps([]),
            "embedding_promedio_json": json.dumps([]),
        })

# 2. Convertir a DataFrame y unir con los datos originales
df_embeddings = pd.DataFrame(records)
df_resultado = pd.concat([df.reset_index(drop=True), df_embeddings], axis=1)

# 3. Guardar en CSV
output_path = "reportes_con_embeddings.csv"
df_resultado.to_csv(output_path, index=False)

print(f"\n¡Extracción finalizada! Guardado en: {output_path}")
print(df_resultado[["idioma", "n_oraciones"]].head())

In [ ]:
df_resultado

In [ ]:
ANCHORS_BY_LANG = {
    "en": [
        "acute knee fracture, cortical break, bone fracture, insufficiency fracture",
        "no evidence of fracture, intact bones, without acute fracture, no osseous lesions",
    ],
    "es": [
        "presencia de fractura ósea, fisura en la rodilla",
        "sin signos de fractura ósea",
    ],
    "tr": [
        "diz ekleminde kırık bulgusu, kortikal kemik kırığı",
        "kırık saptanmadı, kortikal bütünlük korunmuştur, kırık hattı izlenmedi",
    ],
    "bg": [
        "данни за фрактура на коляното, костна фрактура, кортикално прекъсване",
        "няма данни за фрактура, интактни кости, без данни за фрактура",
    ],
    "ru": [
        "признаки перелома коленного сустава, кортикальное нарушение",
        "без признаков перелома, кости интактны, переломов не выявлено",
    ],
    "el": [
        "ευρήματα κατάγματος γόνατος, κάταγμα οστού",
        "χωρίς ενδείξεις κατάγματος, ανέπαφα οστά, απουσία κατάγματος",
    ],
    "de": [
        "Nachweis einer Knochenfraktur des Knies, Fissur",
        "kein Nachweis einer Fraktur, intakte Knochenstrukturen, frakturfrei",
    ],
    "hr": [
        "znakovi frakture koljena, prelom kosti",
        "nema znakova frakture, kosti su intaktne, bez znakova preloma",
    ],
}

# 3. Generar los Embeddings de las Anclas (anchor_vectors)
print("Generando vectores de anclas con OpenAI...")
anchor_vectors = {}
for lang, (pos_text, neg_text) in ANCHORS_BY_LANG.items():
    resp = client.embeddings.create(
        input=[pos_text, neg_text],
        model=MODEL_NAME
    )
    anchor_vectors[lang] = {
        "pos": np.array(resp.data[0].embedding),
        "neg": np.array(resp.data[1].embedding),
    }
print("¡Vectores de anclas precalculados con éxito!\n")

In [ ]:
# 4. Función de evaluación vectorial (Modificada con threshold de confianza)
def evaluar_fractura_estudio(
    embeddings_json, oraciones_json, lang, anchor_dict, margin_ratio=0.90, max_pos_dist=1.15, threshold_confianza=0.6
):
    """Aplica operaciones vectoriales y devuelve la clasificación, score y oración gatillo."""
    if pd.isna(embeddings_json) or not embeddings_json or embeddings_json == '[]':
        return 0, 0.0, "Sin oraciones"

    # Deserializar matrices y textos
    vecs = np.array(json.loads(embeddings_json)) if isinstance(embeddings_json, str) else np.array(embeddings_json)
    oraciones = json.loads(oraciones_json) if isinstance(oraciones_json, str) else oraciones_json

    if vecs.ndim != 2 or vecs.shape[0] == 0 or len(oraciones) == 0:
        return 0, 0.0, "Sin oraciones válidas"

    # Seleccionar ancla según idioma (fallback a inglés)
    target_lang = lang if lang in anchor_dict else 'en'
    v_pos = anchor_dict[target_lang]['pos']
    v_neg = anchor_dict[target_lang]['neg']

    # --- Operaciones Vectoriales Vectorizadas (Broadcasting) ---
    dists_pos = np.sum((vecs - v_pos) ** 2, axis=1)
    dists_neg = np.sum((vecs - v_neg) ** 2, axis=1)

    # Filtros base
    polaridad_ok = dists_pos < (margin_ratio * dists_neg)
    relevancia_ok = dists_pos < max_pos_dist
    mascara_positiva = polaridad_ok & relevancia_ok

    if np.any(mascara_positiva):
        # Buscar el índice de la oración que más se acercó a fractura
        indices_positivos = np.where(mascara_positiva)[0]
        indice_ganador = indices_positivos[np.argmin(dists_pos[indices_positivos])]

        # Calcular la confianza
        best_score = 1.0 - (dists_pos[indice_ganador] / 2.0)

        # === APLICACIÓN DEL UMBRAL DE CONFIANZA ===
        if best_score > threshold_confianza:
            return 1, float(best_score), oraciones[indice_ganador]
        else:
            # Pasó los filtros iniciales, pero no alcanzó el umbral estricto (0.6)
            indice_negativo_fuerte = np.argmin(dists_neg)
            return 0, float(best_score), oraciones[indice_negativo_fuerte]

    else:
        # Es NEGATIVO absoluto: Devolver la oración que más asegura que NO hay fractura
        indice_negativo_fuerte = np.argmin(dists_neg)
        return 0, 0.0, oraciones[indice_negativo_fuerte]


# 5. Aplicar la lógica sobre el DataFrame
print("Clasificando reportes y extrayendo justificaciones...")

# Hacemos zip de 3 columnas: los vectores, los textos de las oraciones y el idioma
resultados = [
    evaluar_fractura_estudio(
        emb, orac, lang, anchor_vectors,
        margin_ratio=0.90,
        max_pos_dist=1.15,
        threshold_confianza=0.60  # <--- AQUÍ PUEDES MODIFICAR TU UMBRAL
    )
    for emb, orac, lang in tqdm(
        zip(
            df_resultado['embeddings_oraciones_json'],
            df_resultado['oraciones_json'],
            df_resultado['idioma']
        ),
        total=len(df_resultado)
    )
]

# Guardar resultados expandidos en el DataFrame
df_resultado['pred_fractura_binaria'] = [r[0] for r in resultados]
df_resultado['confianza_fractura'] = [r[1] for r in resultados]
df_resultado['oracion_justificacion'] = [r[2] for r in resultados]

print('\nDistribución final de la suposición:')
print(df_resultado['pred_fractura_binaria'].value_counts(normalize=True))

# Muestra un vistazo de por qué clasificó de esa forma
print("\nEjemplos de justificaciones:")
print(df_resultado[['pred_fractura_binaria', 'confianza_fractura', 'oracion_justificacion', "Fracture"]].head(10))

In [ ]:
df_resultado.info()

In [ ]:
df_resultado[['pred_fractura_binaria', 'oracion_justificacion', "confianza_fractura", "Fracture"]].to_csv("/content/reportes_resultado.csv")

In [ ]:
from google.colab import files
files.download("/content/reportes_resultado.csv")

In [ ]:
df_resultado[['pred_fractura_binaria', 'oracion_justificacion', "confianza_fractura"]]

In [ ]:
train_etiquetas.info()

In [ ]:
import numpy as np

In [ ]:
train_series = np.where(train_series_csv["StudyInstanceUID"] == train_etiquetas.loc[1027]["StudyInstanceUID"])

In [ ]:
train_series_csv[train_series_csv["StudyInstanceUID"] == train_etiquetas.loc[1027]["StudyInstanceUID"]]

In [ ]:
train_etiquetas.loc[1027]["StudyInstanceUID"]

https://python-graph-gallery.com/web-bubble-plot-with-annotations-and-custom-features/

In [ ]:
# Libraries
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:

df1 = pd.read_csv("/content/reportes_resultado (1).csv")
df2 = pd.read_csv("/content/reportes_resultado.csv")


In [ ]:
df2

In [ ]:
df_merged =  pd.concat([df1, df2], axis=0, ignore_index=True)

In [ ]:
df_merged

In [ ]:
import seaborn as sns
import matplotlib.patches as mpatches
# Valor de ejemplo para punt_AUC (reemplazar por tu cálculo real, e.g., roc_auc_score(df['Fracture'], df['confianza_fractura']))
punt_AUC = 0.6764

plt.figure(figsize=(10, 7))

# 1. Dibujar el Scatter Plot con Jitter
ax = sns.stripplot(
    data=df_merged,
    x='Fracture',
    y='confianza_fractura',
    hue='pred_fractura_binaria',
    palette={0: '#1f77b4', 1: '#d62728'},
    jitter=0.3,
    alpha=0.7,
    size=7,
    edgecolor="white",
    linewidth=0.5
)

# 2. Configurar los límites
plt.xlim(-0.5, 1.5)
plt.ylim(0, 1.05)

# 3. Dibujar las líneas divisorias
plt.axhline(y=0.60, color='black', linestyle='--', linewidth=2)
plt.axvline(x=0.5, color='black', linestyle='-', linewidth=2)

# 4. Sombrear las 4 regiones
plt.axhspan(0, 0.60, xmin=0, xmax=0.5, facecolor='lightgreen', alpha=0.15)
plt.axhspan(0.60, 1.05, xmin=0, xmax=0.5, facecolor='lightcoral', alpha=0.15)
plt.axhspan(0, 0.60, xmin=0.5, xmax=1, facecolor='lightcoral', alpha=0.15)
plt.axhspan(0.60, 1.05, xmin=0.5, xmax=1, facecolor='lightgreen', alpha=0.15)

# 5. Agregar Etiquetas de Texto para TP, FP, TN, FN
plt.text(-0.4, 0.95, 'Falsos Positivos (FP)\nError', fontsize=12, color='darkred', weight='bold')
plt.text(-0.4, 0.05, 'Verdaderos Negativos (TN)\nAcierto', fontsize=12, color='darkgreen', weight='bold')
plt.text(0.6, 0.95, 'Verdaderos Positivos (TP)\nAcierto', fontsize=12, color='darkgreen', weight='bold')
plt.text(0.6, 0.05, 'Falsos Negativos (FN)\nError', fontsize=12, color='darkred', weight='bold')

# 6. Formato de ejes y título
plt.xticks([0, 1], ['0.0 (No Fractura)', '1.0 (Sí Fractura)'], fontsize=12, weight='bold')
plt.xlabel('Ground Truth en la Imagen', fontsize=13)
plt.ylabel('Confianza del Embedding', fontsize=13)
plt.title('Matriz de Confusión: Reporte vs Realidad', fontsize=15, weight='bold')

# 7. Configurar leyenda incluyendo punt_AUC
handles, labels = ax.get_legend_handles_labels()
auc_proxy = mpatches.Patch(color='none', label=f'AUC = {punt_AUC:.3f}')
handles.append(auc_proxy)

plt.legend(
    handles=handles,
    title='Suposición del Código',
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    frameon=True
)

plt.tight_layout()

# 8. Guardar en formato PNG antes de mostrar
plt.savefig('matriz_confusion_confianza.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
files.download('matriz_confusion_confianza.png')

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

# Asegúrate de limpiar NaNs en caso de que algún cruce haya fallado
df_evaluacion_clean = df_merged.dropna(subset=['Fracture', 'confianza_fractura'])

# 1. Calcular el AUC Score
auc_score = roc_auc_score(
    y_true=df_evaluacion_clean['Fracture'],
    y_score=df_evaluacion_clean['confianza_fractura']
)

print(f"Puntaje AUC (Area Under Curve): {auc_score:.4f}")

# ==========================================
# 2. Graficar la Curva ROC (Opcional pero recomendado)
# ==========================================
# Calcular los puntos de la curva: Tasa de Falsos Positivos (FPR) vs Tasa de Verdaderos Positivos (TPR)
fpr, tpr, thresholds = roc_curve(
    y_true=df_evaluacion_clean['Fracture'],
    y_score=df_evaluacion_clean['confianza_fractura']
)

plt.figure(figsize=(7, 6))

# Dibujar la curva ROC de nuestro modelo
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Modelo (AUC = {auc_score:.3f})')

# Dibujar la línea de azar (y=x)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Azar (AUC = 0.500)')

# Formato del gráfico
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
plt.title('Curva ROC: Detección de Fractura', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()

In [ ]:
!pip install opendatasets
!pip install pandas

In [ ]:
!pip install legacy-cgi

In [ ]:
!pip install kaggle

Analizamos los datos de manera automatica

In [ ]:
!pip install ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport

In [ ]:
 # Generate the profile report
profile = ProfileReport(train, title="Analisis automatico del train dataframe")

# Display the report in a Jupyter Notebook
profile.to_notebook_iframe()

In [ ]:
 # Generate the profile report
profile = ProfileReport(train_series_csv, title="Analisis automatico del train dataframe")

# Display the report in a Jupyter Notebook
profile.to_notebook_iframe()

Mergeamos los dataframes en base a la columna "StudyInstanceUID"

In [ ]:
train_merged = train.merge(train_series_csv, on="StudyInstanceUID")

In [ ]:
train_merged.describe(include="all")

In [ ]:
train_merged.info()

In [ ]:
 # Generate the profile report
profile = ProfileReport(train_merged, title="Analisis automatico del train dataframe")

# Display the report in a Jupyter Notebook
profile.to_notebook_iframe()

In [ ]:
train_merged.columns

SMOTE N categorical VARIATION


```
https://www.geeksforgeeks.org/machine-learning/smote-for-imbalanced-classification-with-python/
```



In [ ]:
X_train = train_merged.drop(columns=['ACL', 'MCL', 'Medial Meniscus',
       'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
       'Synovitis', "Baker's", 'Contusion', 'Fracture', 'SeriesInstanceUID',
       'Fluid_Sensitive', 'Fat_Suppression', 'Anatomical_Plane'])
y_train = train_merged.drop(columns=['StudyInstanceUID', 'Report'])

Vectorizar con text embedding

In [ ]:
from imblearn.over_sampling import ADASYN

adasyn = ADASYN(sampling_strategy='minority', random_state=42)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)

print("Class distribution AFTER ADASYN:")
print(y_train_adasyn.value_counts())

Subida del codigo a Github

In [ ]:
!git config  --global init.Defaultbranch master # Cargamos la rama default
TOKEN = ""
owner = ""
!git config --global user.email ""
!git config --global user.name ""
pathurl = f"https://{TOKEN}@github.com/{owner}/Proyectoanalitica.git"

# Limpiamos y reiniciamos
!git remote remove origin 2>/dev/null
!git init

# Iniciamos la subida
!git add archivo.txt
!git commit -m "Primer commit"
!git remote add origin {pathurl}
# Usamos la variable pathurl directamente en el push para asegurar el token
!git push -f origin master
!git pull